[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/nn_exercise_covertype.ipynb)

# Neural networks from scratch to PyTorch: forest cover type

This is an **advanced** exercise. You will:

* **Part A:** implement a neural network with one hidden layer yourself, with NumPy (forward pass, softmax, cross-entropy and **backpropagation**), check your gradients numerically, and see why a hidden layer can learn what logistic regression cannot;
* **Part B:** prepare a real dataset, the [Covertype dataset](https://archive.ics.uci.edu/dataset/31/covertype): predict which of **7 types of forest** covers a 30 × 30 m patch of land in Colorado, from cartographic data only (elevation, slope, distances, soil type, ...). You will also build the baselines that the network has to beat;
* **Part C:** build and train a multilayer perceptron in **PyTorch**, starting by **debugging a training loop written by an AI assistant**;
* **Part D:** study **capacity, overfitting and regularisation** (dropout, weight decay, early stopping) and **class imbalance**;
* **Part E:** evaluate on the test set once and interpret the final model.

### How this notebook works

* Cells with `...` or `# TODO` are for you to complete.
* **Predict first** cells: write what you *expect* **before** running the code. You are graded on your reasoning, not on whether the guess was right.
* **Explain** cells: answer in your own words, using **your own numbers**. Your data subset and your splits depend on your student number, so your results differ from your colleagues'.
* `check(...)` cells test your work. Do not move on until they pass.
* Part A may only use NumPy: `sklearn` is used only to *compare* your results.

### About AI assistants

You may use an AI assistant to help you write code. But notice that most of the work in this notebook is **not** writing code: it is predicting, checking and explaining what the code does. In Part C you will see that code that "looks right" and runs without errors can still be wrong in ways that only show up in the numbers. Whatever tool you use, **you** are responsible for every line: you may be asked to explain or change any cell in class. Declare your use of AI at the end of the notebook.

## 0. Your student number

In [ ]:
STUDENT_ID = None  # TODO: replace with your student number, e.g. 12345

assert isinstance(STUDENT_ID, int), "Set STUDENT_ID to your student number (an integer)."
SEED = STUDENT_ID % (2**32)

In [ ]:
%matplotlib inline
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


def check(condition, message):
    if not condition:
        raise AssertionError("❌ " + message)
    print("✅", message)


np.random.seed(SEED)
torch.manual_seed(SEED);

# Part A: a neural network with NumPy

We will build a network with **one hidden layer** of $h$ ReLU units and a **softmax** output layer with $K$ units (one per class). With $m$ examples stacked as the rows of $X \in \mathbb{R}^{m \times n}$, the **forward pass** is

$$Z_1 = X W_1 + b_1 \qquad A_1 = \mathrm{ReLU}(Z_1) = \max(0, Z_1) \qquad Z_2 = A_1 W_2 + b_2 \qquad P = \mathrm{softmax}(Z_2)$$

with $W_1 \in \mathbb{R}^{n \times h}$, $b_1 \in \mathbb{R}^{h}$, $W_2 \in \mathbb{R}^{h \times K}$, $b_2 \in \mathbb{R}^{K}$. The softmax turns each row of scores into probabilities:

$$P_{ik} = \frac{e^{Z_{ik}}}{\sum_{j=1}^{K} e^{Z_{ij}}}$$

The **cross-entropy** cost, with the labels $y_i \in \{0, \dots, K-1\}$, is the mean of minus the log-probability given to the correct class:

$$J = -\frac{1}{m} \sum_{i=1}^{m} \log P_{i, y_i}$$

**Backpropagation** applies the chain rule from the output back to the input. With $Y$ the one-hot matrix of the labels ($Y_{ik} = 1$ if $y_i = k$):

$$dZ_2 = \frac{1}{m}(P - Y) \qquad dW_2 = A_1^T dZ_2 \qquad db_2 = \textstyle\sum_i dZ_2$$
$$dA_1 = dZ_2 W_2^T \qquad dZ_1 = dA_1 \odot \mathbb{1}[Z_1 > 0] \qquad dW_1 = X^T dZ_1 \qquad db_1 = \textstyle\sum_i dZ_1$$

where $dW$ means $\frac{\partial J}{\partial W}$, $\odot$ is the element-wise product and the sums are over the rows (the examples).

Implement the functions below **without loops over the examples**. The initialisation is given.

In [ ]:
def init_params(n_in, n_hidden, n_out, rng):
    # "He" initialisation: random weights with a variance adapted to ReLU units
    return {
        "W1": rng.randn(n_in, n_hidden) * np.sqrt(2 / n_in),
        "b1": np.zeros(n_hidden),
        "W2": rng.randn(n_hidden, n_out) * np.sqrt(2 / n_hidden),
        "b2": np.zeros(n_out),
    }


def relu(z):
    ...  # TODO


def softmax(z):
    # z has shape (m, K); return an (m, K) matrix whose rows sum to 1
    ...  # TODO


def cross_entropy(P, y):
    # P: (m, K) probabilities, y: (m,) integer labels
    ...  # TODO


def forward(params, X):
    # Return P and a cache with the values that backward needs
    ...  # TODO
    return P, (X, Z1, A1)


def backward(params, cache, P, y):
    X, Z1, A1 = cache
    m = len(y)
    ...  # TODO
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

### A.1 Testing your functions

The next cell checks your functions. The last check compares your gradients with a **numerical approximation**: for every parameter $w$ and a small $\varepsilon$,

$$\frac{\partial J}{\partial w} \approx \frac{J(w + \varepsilon) - J(w - \varepsilon)}{2\varepsilon}$$

If your backpropagation has a bug, this is the check that finds it.

In [ ]:
check(np.allclose(relu(np.array([-2.0, 0.0, 3.0])), [0, 0, 3]), "relu")
P_test = softmax(np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]]))
check(P_test.shape == (2, 3) and np.allclose(P_test.sum(axis=1), 1), "softmax rows sum to 1")
check(np.allclose(P_test[1], 1 / 3), "softmax of equal scores is uniform")
check(np.allclose(softmax(np.array([[1000.0, 1001.0]])), softmax(np.array([[0.0, 1.0]]))), "softmax works with large scores")
check(np.isclose(cross_entropy(np.full((4, 5), 0.2), np.array([0, 1, 2, 3])), np.log(5)), "cross-entropy of a uniform prediction is log(K)")

rng = np.random.RandomState(SEED)
Xg, yg = rng.randn(6, 3), rng.randint(0, 4, size=6)
pg = init_params(3, 5, 4, rng)
P, cache = forward(pg, Xg)
check(P.shape == (6, 4), "forward returns one row of probabilities per example")
grads = backward(pg, cache, P, yg)

eps, max_error = 1e-6, 0
for name in pg:
    for idx in np.ndindex(pg[name].shape):
        old = pg[name][idx]
        pg[name][idx] = old + eps
        j_plus = cross_entropy(forward(pg, Xg)[0], yg)
        pg[name][idx] = old - eps
        j_minus = cross_entropy(forward(pg, Xg)[0], yg)
        pg[name][idx] = old
        max_error = max(max_error, abs((j_plus - j_minus) / (2 * eps) - grads[name][idx]))
print("largest difference between numerical and analytical gradient:", max_error)
check(max_error < 1e-6, "backpropagation matches the numerical gradient")

**Explain:**
1. Why does `softmax` subtract the maximum of each row before the exponential? Why does it not change the result? (The test with `[1000, 1001]` is a hint.)
2. Gradient checking is very slow: count how many times `forward` is called in the loop above. Why do we use it only to **test** the code, and never to train?

**Your answer:**

### A.2 A problem that logistic regression cannot solve

The function below generates a 2D dataset of three interleaved **spirals** (one class per spiral). It is given, together with a helper that plots the decision regions of any classifier.

In [ ]:
def make_spirals(n_per_class, n_classes, rng):
    X, y = [], []
    for k in range(n_classes):
        r = np.linspace(0.05, 1, n_per_class)
        t = np.linspace(k * 4, (k + 1) * 4, n_per_class) + rng.randn(n_per_class) * 0.2
        X.append(np.c_[r * np.sin(t), r * np.cos(t)])
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)


def plot_regions(predict, X, y, title, ax):
    xx, yy = np.meshgrid(np.linspace(-1.1, 1.1, 300), np.linspace(-1.1, 1.1, 300))
    zz = predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, cmap="viridis")
    ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="viridis", edgecolor="k", linewidth=0.3)
    ax.set_title(title)


X_sp, y_sp = make_spirals(150, 3, np.random.RandomState(SEED))
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_sp[:, 0], X_sp[:, 1], c=y_sp, s=12, cmap="viridis", edgecolor="k", linewidth=0.3);

**Predict first:** Logistic regression (with the softmax, for 3 classes) computes one **linear** score per class. What will its decision regions look like on the spirals, and roughly what accuracy do you expect? And a network with one hidden layer of 100 ReLU units?

**Your answer:**

Complete `train_numpy`: **batch gradient descent**, using your `forward`, `cross_entropy` and `backward`. In each iteration update every parameter with $w := w - \alpha \frac{\partial J}{\partial w}$ and store the cost.

In [ ]:
def train_numpy(params, X, y, lr=1.0, n_iters=3000):
    history = []
    for _ in range(n_iters):
        ...  # TODO
    return params, history


def predict_numpy(params, X):
    return forward(params, X)[0].argmax(axis=1)

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg_sp = LogisticRegression().fit(X_sp, y_sp)
params_sp, hist_sp = train_numpy(init_params(2, 100, 3, np.random.RandomState(SEED)), X_sp, y_sp)

acc_logreg_sp = np.mean(logreg_sp.predict(X_sp) == y_sp)
acc_mlp_sp = np.mean(predict_numpy(params_sp, X_sp) == y_sp)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_regions(logreg_sp.predict, X_sp, y_sp, f"logistic regression: {acc_logreg_sp:.1%}", axes[0])
plot_regions(lambda Z: predict_numpy(params_sp, Z), X_sp, y_sp, f"100 hidden units: {acc_mlp_sp:.1%}", axes[1])
axes[2].plot(hist_sp)
axes[2].set_xlabel("iteration")
axes[2].set_ylabel("cost J")
axes[2].set_title("training of the network");

In [ ]:
check(all(b <= a + 1e-3 for a, b in zip(hist_sp, hist_sp[1:])), "the cost (almost) never increases")
check(acc_mlp_sp > 0.95, "the network learns the spirals (training accuracy > 95%)")
check(acc_logreg_sp < 0.7, "logistic regression cannot learn them")

**Explain:** Was your prediction right? Each hidden ReLU unit computes $\max(0, w^T x + b)$. Use that to explain the shape of the network's decision boundary (zoom in on the plot: is it smooth?). Why would a network with a hidden layer but **no** activation function (or with the identity as activation) draw the same boundaries as logistic regression?

**Your answer:**

### A.3 Why random initialisation?

`init_params` initialises the weights at random. What if we gave **every** weight the same value?

**Predict first:** Initialise all weights to 0.5 (and the biases to 0) and train on the spirals. What will happen to the 100 hidden units during training? What accuracy do you expect?

**Your answer:**

In [ ]:
params_sym = {"W1": np.full((2, 100), 0.5), "b1": np.zeros(100),
              "W2": np.full((100, 3), 0.5), "b2": np.zeros(3)}
params_sym, hist_sym = train_numpy(params_sym, X_sp, y_sp)
acc_sym = np.mean(predict_numpy(params_sym, X_sp) == y_sp)
print(f"training accuracy: {acc_sym:.1%}")
print("all hidden units have the same weights:", np.allclose(params_sym["W1"], params_sym["W1"][:, :1]))

**Explain:** Explain the result using the backpropagation equations: why do all the columns of $W_1$ receive the same update in every iteration? Why does **random** initialisation solve this?

**Your answer:**

# Part B: the Covertype dataset

### B.1 Load

The data comes from the US Forest Service. Each row is a 30 × 30 m cell of the Roosevelt National Forest (Colorado), described by:

| column(s) | meaning |
|---|---|
| `Elevation` | elevation (m) |
| `Aspect` | direction the slope faces, in degrees (0 to 360, 0 = north) |
| `Slope` | slope (degrees) |
| `Horizontal_Distance_To_Hydrology`, `Vertical_Distance_To_Hydrology` | distance to the nearest water (m) |
| `Horizontal_Distance_To_Roadways` | distance to the nearest road (m) |
| `Hillshade_9am`, `Hillshade_Noon`, `Hillshade_3pm` | amount of sun light at the summer solstice (0 to 255) |
| `Horizontal_Distance_To_Fire_Points` | distance to the nearest wildfire ignition point (m) |
| `Wilderness_Area_0` ... `Wilderness_Area_3` | one-hot: which of the 4 wilderness areas |
| `Soil_Type_0` ... `Soil_Type_39` | one-hot: which of the 40 soil types |
| `Cover_Type` | **target**: the dominant tree species (7 classes) |

The full dataset has 581,012 rows. To keep training times reasonable, each student works with a different random sample of 40,000 rows. The first run downloads the data (≈ 11 MB) and may take about a minute; after that it is read from a local cache.

In [ ]:
from sklearn.datasets import fetch_covtype

raw = fetch_covtype(as_frame=True).frame
df = raw.sample(n=40_000, random_state=SEED).reset_index(drop=True)

CLASSES = ["Spruce/Fir", "Lodgepole Pine", "Ponderosa Pine", "Cottonwood/Willow", "Aspen", "Douglas-fir", "Krummholz"]
NUMERIC = list(df.columns[:10])
df["Cover_Type"] = df["Cover_Type"] - 1  # classes 1..7 become 0..6
df.head()

### B.2 Exploration

1. Check the dimensions and the types of the columns, and whether there are missing values.
2. Plot the number of examples of each class (use the names in `CLASSES` for the labels).
3. Plot the distribution of `Elevation` for each class (e.g. a [`boxplot`](https://seaborn.pydata.org/generated/seaborn.boxplot.html) with `x="Cover_Type"`, `y="Elevation"`).
4. Check that every row belongs to **exactly one** wilderness area and **exactly one** soil type.

In [ ]:
# TODO: shape, types, missing values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
# TODO: plot 2 on axes[0] and plot 3 on axes[1]

In [ ]:
wilderness_cols = [c for c in df.columns if c.startswith("Wilderness")]
soil_cols = [c for c in df.columns if c.startswith("Soil")]
one_wilderness = ...  # TODO: True if every row has exactly one wilderness area
one_soil = ...        # TODO

In [ ]:
check(one_wilderness and one_soil, "every row has exactly one wilderness area and one soil type")
print("soil types that never appear in your sample:", [c for c in soil_cols if df[c].sum() == 0])

**Explain:**
1. Is the dataset balanced? What accuracy would a "model" that always predicts the most frequent class get **on your data**? Which classes will be hardest to learn, simply because of the number of examples?
2. Which classes can be (almost) separated using `Elevation` alone, and which ones overlap?
3. `Aspect` is an angle: 1° and 359° are almost the same direction. Why is the raw value a poor input for a model? Suggest a better encoding.

**Your answer:**

### B.3 Split

We will compare many models and hyperparameters, so we need **three** sets. Split the data into 60% **train**, 20% **validation** and 20% **test**, using `random_state=SEED` and `stratify` in both splits (hint: call `train_test_split` twice: first separate the test set, then split the rest in 75% / 25%).

From now until Part E, **the test set is locked**: do not use `X_test` or `y_test` for anything.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="Cover_Type")
y = df["Cover_Type"].to_numpy()

X_train, X_val, X_test, y_train, y_val, y_test = ...  # TODO

In [ ]:
check((len(X_train), len(X_val), len(X_test)) == (24_000, 8_000, 8_000), "60% / 20% / 20%")
check(np.allclose(np.bincount(y_train) / len(y_train), np.bincount(y_val, minlength=7) / len(y_val), atol=0.005), "the splits are stratified")

**Explain:** In the previous exercises we used cross-validation on the training set to choose hyperparameters. Here we use a single validation set instead. Why is that a reasonable choice for neural networks on this dataset? What do we lose?

**Your answer:**

### B.4 Scaling

**Predict first:** We will standardise the 10 numeric columns (`NUMERIC`). Should we also standardise the 44 one-hot columns? Think about what happens to a column that is always 0 in the training set if you compute $(x - \mu) / \sigma$ yourself.

**Your answer:**

Complete `prepare`: it returns a `float32` NumPy array where the `NUMERIC` columns are standardised with a [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) **fitted on the training set only**, and the one-hot columns are unchanged.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(...)  # TODO


def prepare(X):
    X = X.copy()
    ...  # TODO
    return X.to_numpy(dtype=np.float32)


A_train, A_val, A_test = prepare(X_train), prepare(X_val), prepare(X_test)

In [ ]:
check(A_train.dtype == np.float32 and A_train.shape == (24_000, 54), "A_train is a float32 matrix with 54 columns")
check(np.allclose(A_train[:, :10].mean(axis=0), 0, atol=1e-4) and np.allclose(A_train[:, :10].std(axis=0), 1, atol=1e-3), "numeric training columns are standardised")
check(not np.allclose(A_val[:, :10].mean(axis=0), 0, atol=1e-4), "the scaler was not fitted on the validation set")
check(set(np.unique(A_train[:, 10:])) <= {0, 1}, "one-hot columns are unchanged")

### B.5 Baselines

Before training a neural network we need to know what "good" means. The helper `record` stores the **accuracy** and the **balanced accuracy** (the mean of the recall of each class, so every class counts the same) of a model in the dictionary `results`.

Record two baselines on the **validation** set:

1. `"majority"`: always predict the most frequent class of the **training** set;
2. `"logistic regression"`: `LogisticRegression(max_iter=2000)` trained on `A_train`.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

results = {}


def record(name, y_true, y_pred):
    results[name] = {"accuracy": accuracy_score(y_true, y_pred),
                     "balanced accuracy": balanced_accuracy_score(y_true, y_pred)}
    return pd.DataFrame(results).T.round(4)


majority_class = ...  # TODO
record("majority", y_val, np.full(len(y_val), majority_class))

logreg = ...  # TODO
record("logistic regression", y_val, logreg.predict(A_val))

In [ ]:
check(np.isclose(results["majority"]["balanced accuracy"], 1 / 7), "the majority baseline has a balanced accuracy of 1/7")
check(results["logistic regression"]["accuracy"] > results["majority"]["accuracy"] + 0.15, "logistic regression beats the majority baseline")

**Explain:** Why is the balanced accuracy of the majority baseline exactly $1/7$? What does the gap between the accuracy and the balanced accuracy of the logistic regression tell you?

**Your answer:**

# Part C: a multilayer perceptron in PyTorch

### C.1 Tensors and data loaders

Create the tensors for the three sets, and `DataLoader`s for the training (batches of 256, shuffled) and validation (batches of 4096, not shuffled) sets.

**Careful with the types:** `nn.CrossEntropyLoss` expects the inputs as `float32` and the labels as **`int64`** (`torch.long`). Check what type `y_train` has now.

In [ ]:
print("y_train has type", y_train.dtype)

Xtr_t = ...  # TODO
ytr_t = ...  # TODO
Xva_t, yva_t = ..., ...  # TODO
Xte_t, yte_t = ..., ...  # TODO

train_ds = TensorDataset(Xtr_t, ytr_t)
val_ds = TensorDataset(Xva_t, yva_t)
train_loader = ...  # TODO
val_loader = ...    # TODO

In [ ]:
check(Xtr_t.dtype == torch.float32 and ytr_t.dtype == torch.long and yva_t.dtype == torch.long and yte_t.dtype == torch.long, "inputs are float32 and labels are int64")
check(len(train_loader) == 94 and len(val_loader) == 2, "loaders have the right batch sizes")

### C.2 The model

Complete `make_mlp`: it returns an [`nn.Sequential`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) with, for every size in `hidden`, a `nn.Linear` followed by a `nn.ReLU` (and an `nn.Dropout(dropout)` if `dropout > 0`), and a final `nn.Linear` with `n_out` outputs, **without** activation.

In [ ]:
def make_mlp(n_in, hidden, n_out, dropout=0.0):
    layers = []
    ...  # TODO
    return nn.Sequential(*layers)

In [ ]:
def n_params(model):
    return sum(p.numel() for p in model.parameters())


print(make_mlp(54, [128, 128], 7, dropout=0.2))
check(make_mlp(54, [128, 128], 7)(Xtr_t[:5]).shape == (5, 7), "the network returns 7 scores per example")
check(n_params(make_mlp(54, [128, 128], 7)) == 24_455, "make_mlp(54, [128, 128], 7) has 24,455 parameters")
check(n_params(make_mlp(54, [], 7)) == 385, "make_mlp(54, [], 7) has 385 parameters")
check(sum(isinstance(l, nn.Dropout) for l in make_mlp(54, [64, 64], 7, dropout=0.5)) == 2, "one dropout per hidden layer")

**Explain:** Show how the 24,455 parameters are calculated. Which model from the previous exercises is `make_mlp(54, [], 7)`? Why must the last layer have no softmax?

**Your answer:**

### C.3 Debugging a training loop written by an AI

A student asked an AI assistant: *"write a PyTorch function that trains a classifier for one epoch and returns the loss, and a function that evaluates it and returns the loss and the accuracy"*. This is the answer:

In [ ]:
# Code generated by an AI assistant
def train_epoch(model, loader, loss_fn, optimizer):
    """Trains the model for one epoch and returns the mean training loss."""
    model.train()
    for xb, yb in loader:
        logits = model(xb)
        probs = torch.softmax(logits, dim=1)
        loss = loss_fn(probs, yb)
        loss.backward()
        optimizer.step()
    return loss.item()


def evaluate(model, loader, loss_fn):
    """Returns the mean loss and the accuracy of the model on a dataset."""
    total_loss, correct = 0.0, 0
    for xb, yb in loader:
        logits = model(xb)
        total_loss += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(dim=1) == yb).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n

The code runs without errors, and if you train with it the loss goes down and the accuracy goes up. But it has **four bugs** (one of them is two missing lines that belong together).

**Predict first:** Before running anything, read the code and list the bugs you can find. For each one, say what you expect it to do to the training.

**Your answer:**

Run the checks below with the code as it is: they should **fail**. Then fix the code in the cell above (keep the same function names) until all the checks pass. Each check was designed to catch at least one of the bugs: read it and understand what it tests.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# 1. With lr = 0 the model does not change, so the loss of the epoch must be the mean loss over the dataset
torch.manual_seed(SEED)
model = make_mlp(54, [32], 7)
full_loader = DataLoader(train_ds, batch_size=500, shuffle=True)
epoch_loss = train_epoch(model, full_loader, loss_fn, torch.optim.SGD(model.parameters(), lr=0.0))
check(np.isclose(epoch_loss, evaluate(model, full_loader, loss_fn)[0], rtol=1e-4), "train_epoch returns the mean cross-entropy of the epoch")

# 2. Evaluating the same model twice must give the same result, even with dropout
torch.manual_seed(SEED)
model = make_mlp(54, [64], 7, dropout=0.5)
check(evaluate(model, val_loader, loss_fn) == evaluate(model, val_loader, loss_fn), "evaluate is deterministic (dropout is off)")
check(not model.training, "evaluate leaves the model in evaluation mode")
with torch.enable_grad():
    _ = evaluate(model, val_loader, loss_fn)
    check(all(p.grad is None for p in model.parameters()), "evaluate does not compute gradients")

# 3. A network with 24,455 parameters must be able to memorise 256 examples
torch.manual_seed(SEED)
tiny_loader = DataLoader(TensorDataset(Xtr_t[:256], ytr_t[:256]), batch_size=64, shuffle=True)
model = make_mlp(54, [128, 128], 7)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(100):
    train_epoch(model, tiny_loader, loss_fn, optimizer)
tiny_acc = evaluate(model, tiny_loader, loss_fn)[1]
print(f"accuracy on the 256 training examples: {tiny_acc:.1%}")
check(tiny_acc > 0.99, "the network can memorise a small subset (overfitting test)")

**Explain:**
1. For each bug, which check caught it, and why? Is there a bug that **no** check would have caught if it were the only one? (Look at what the third `evaluate` check really tests.)
2. Check 3 trains a network on only 256 examples until it memorises them. This is a classic debugging technique for neural networks. Why is it useful, and why does passing it say nothing about how good the model will be on new data?
3. Based on this experience, write two concrete rules for yourself when using AI-generated code for machine learning.

**Your answer:**

### C.4 Training with early stopping

The function `fit` trains a model and records, for every epoch, the training loss, the validation loss and the validation accuracy.

Complete the **early stopping** part: keep a copy of the weights with the lowest validation loss so far (use `copy.deepcopy(model.state_dict())`), stop if the validation loss did not improve for `patience` epochs, and at the end restore the best weights (`model.load_state_dict(...)`). If `patience` is `None` there is no early stopping and the model keeps the weights of the last epoch.

In [ ]:
def fit(model, epochs=30, lr=1e-3, weight_decay=0.0, patience=None, loss_fn=None, verbose=True):
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_loss, best_epoch, best_state = np.inf, 0, None

    for epoch in range(epochs):
        history["train_loss"].append(train_epoch(model, train_loader, loss_fn, optimizer))
        val_loss, val_acc = evaluate(model, val_loader, nn.CrossEntropyLoss())
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f"epoch {epoch:3d}: train loss {history['train_loss'][-1]:.4f}, val loss {val_loss:.4f}, val acc {val_acc:.4f}")

        # TODO: early stopping

    # TODO: restore the best weights (only if patience is not None)
    return history

The helpers below plot the curves of one or more trained models and compute the predictions of a model. Then we train a network with two hidden layers of 128 units, with early stopping.

In [ ]:
def plot_histories(histories):
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
    for name, h in histories.items():
        line, = axes[0].plot(h["train_loss"], "--", label=f"{name} (train)")
        axes[0].plot(h["val_loss"], color=line.get_color(), label=f"{name} (val)")
        axes[1].plot(h["val_acc"], color=line.get_color(), label=name)
    axes[0].set_ylabel("cross-entropy")
    axes[1].set_ylabel("validation accuracy")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.legend()


def predict(model, X_t):
    model.eval()
    with torch.no_grad():
        return model(X_t).argmax(dim=1).numpy()


models, histories = {}, {}

torch.manual_seed(SEED)
models["MLP 2x128"] = make_mlp(54, [128, 128], 7)
histories["MLP 2x128"] = fit(models["MLP 2x128"], epochs=60, patience=5)
record("MLP 2x128", y_val, predict(models["MLP 2x128"], Xva_t))

In [ ]:
h = histories["MLP 2x128"]
check(len(h["val_loss"]) <= 60, "fit stops at most after 60 epochs")
check(np.isclose(evaluate(models["MLP 2x128"], val_loader, nn.CrossEntropyLoss())[0], min(h["val_loss"])), "the model has the weights of the best epoch")
check(results["MLP 2x128"]["accuracy"] > results["logistic regression"]["accuracy"] + 0.04, "the MLP beats logistic regression")
plot_histories({"MLP 2x128": h})

### C.5 The learning rate

**Predict first:** We will train the same network for 15 epochs with learning rates $10^{-4}$, $10^{-3}$, $10^{-2}$ and $10^{-1}$. Sketch (in words) the validation loss curve you expect for each one.

**Your answer:**

Train one `make_mlp(54, [128, 128], 7)` per learning rate (call `torch.manual_seed(SEED)` before creating each model so they all start from the same weights), with `epochs=15`, no early stopping and `verbose=False`. Store each history in `lr_histories` and plot them with `plot_histories`.

In [ ]:
lr_histories = {}
for lr in [1e-4, 1e-3, 1e-2, 1e-1]:
    ...  # TODO

plot_histories(lr_histories)

In [ ]:
check(len(lr_histories) == 4 and all(len(h["val_loss"]) == 15 for h in lr_histories.values()), "four histories of 15 epochs")

**Explain:** Was your prediction right? Which learning rate would you choose for this network, and why not simply the one with the lowest validation loss at epoch 15? (Look at the shape of the curves, not only at the last point.)

**Your answer:**

# Part D: capacity and regularisation

### D.1 A bigger network

**Predict first:** We now train a much bigger network, with three hidden layers of 512 units, for 40 epochs **without** early stopping. How many parameters does it have compared with the 2x128 network? What do you expect to happen to the training loss, the validation loss and the validation accuracy over the 40 epochs?

**Your answer:**

In [ ]:
torch.manual_seed(SEED)
models["MLP 3x512"] = make_mlp(54, [512, 512, 512], 7)
print("parameters:", n_params(models["MLP 3x512"]))
histories["MLP 3x512"] = fit(models["MLP 3x512"], epochs=40)
record("MLP 3x512", y_val, predict(models["MLP 3x512"], Xva_t))
plot_histories({k: histories[k] for k in ["MLP 2x128", "MLP 3x512"]})

**Explain:**
1. Was your prediction right? At which epoch is the validation loss of the big network lowest, and what happens after that?
2. You probably see that the validation **loss** goes up while the validation **accuracy** keeps going up (or stays the same). How can both happen at the same time? (Hint: the cross-entropy of an example is $-\log P_{i,y_i}$. What happens to it when the network is very confident and wrong?)
3. The big network has many more parameters than the small one, yet its validation accuracy is higher. Does this contradict what you learned about overfitting?

**Your answer:**

### D.2 Regularisation

Train the same 3x512 architecture three more times (always `torch.manual_seed(SEED)` before creating the model), store each model and history in `models` / `histories` and `record` its validation results:

1. `"3x512 dropout"`: `dropout=0.3`, 40 epochs, no early stopping;
2. `"3x512 weight decay"`: `weight_decay=1e-4` in `fit` (an L2 penalty on the weights), 40 epochs, no early stopping;
3. `"3x512 early stopping"`: no dropout and no weight decay, `epochs=40`, `patience=5`.

**Predict first:** Which of the three will have the lowest validation loss? And the highest validation accuracy?

**Your answer:**

In [ ]:
# TODO: the three models

In [ ]:
check({"3x512 dropout", "3x512 weight decay", "3x512 early stopping"} <= set(results), "the three models were recorded")
plot_histories({k: histories[k] for k in ["MLP 3x512", "3x512 dropout", "3x512 weight decay", "3x512 early stopping"]})
pd.DataFrame(results).T.round(4)

**Explain:**
1. Compare the curves with those of the unregularised 3x512. How does each technique change the gap between training and validation loss?
2. With dropout, the training loss (dashed line) may be **higher** than the validation loss. Why is that not a bug? (Remember what `model.train()` and `model.eval()` do.)
3. Which model would you choose so far, and based on which metric? Is a difference of a few tenths of a percentage point in validation accuracy meaningful with 8,000 examples?

**Your answer:**

### D.3 Class imbalance

Use [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) (with `target_names=CLASSES`) to see the precision and recall of each class of your best model so far, on the validation set.

In [ ]:
from sklearn.metrics import classification_report

best_so_far = ...  # TODO: the name of your best model in `models`

A common way to handle imbalance is to give each class a **weight** in the loss, inversely proportional to its frequency: errors in rare classes then cost more. `nn.CrossEntropyLoss(weight=w)` accepts a tensor `w` with one weight per class.

**Predict first:** If we train your best configuration with weights $w_k = \frac{m}{K \cdot m_k}$ ($m_k$ = number of training examples of class $k$), what will happen to the **accuracy** and to the **balanced accuracy**? Why?

**Your answer:**

Compute `class_weights` (a `float32` tensor), train your best configuration again with `loss_fn=nn.CrossEntropyLoss(weight=class_weights)`, and record it as `"weighted loss"`. Note that `fit` uses the weighted loss for training, but the **unweighted** loss for validation, so the validation losses can still be compared.

In [ ]:
class_weights = ...  # TODO

torch.manual_seed(SEED)
models["weighted loss"] = ...  # TODO
histories["weighted loss"] = ...  # TODO
record("weighted loss", y_val, predict(models["weighted loss"], Xva_t))

In [ ]:
check(class_weights.shape == (7,) and np.isclose((class_weights.numpy() * np.bincount(y_train)).sum(), len(y_train), rtol=1e-4), "class weights are m / (K m_k)")
print(classification_report(y_val, predict(models["weighted loss"], Xva_t), target_names=CLASSES, digits=3))
pd.DataFrame(results).T.round(4)

**Explain:** Was your prediction right? Look at the recall and the precision of the rarest class (`Cottonwood/Willow`) with and without weights. Give an example of an application of this model where you would prefer the weighted model, and one where you would not.

**Your answer:**

# Part E: final evaluation

Choose **one** final model based **only** on the validation results in `results` (say which metric you use and why). Evaluate it on the test set, **once**, and also evaluate the logistic regression on the test set for comparison.

In [ ]:
print(pd.DataFrame(results).T.sort_values("accuracy"))

final_name = ...  # TODO
final_model = models[final_name]
test_pred = ...   # TODO

final = {"final model": final_name,
         "test accuracy": accuracy_score(y_test, test_pred),
         "test balanced accuracy": balanced_accuracy_score(y_test, test_pred),
         "logistic regression test accuracy": accuracy_score(y_test, logreg.predict(A_test))}
final

Plot the confusion matrix of the final model on the test set, normalised by row (`normalize="true"`, so each row shows the recall of a class), with the class names on the axes.

In [ ]:
from sklearn.metrics import confusion_matrix

# TODO

In [ ]:
check(final_name in models and len(test_pred) == len(y_test), "final model evaluated on the test set")

**Explain:** Is the test accuracy close to the validation accuracy of the chosen model? Which pairs of classes are most confused, and does it make sense given the elevation plot of B.2? How much better than logistic regression is the network, and is the difference worth the extra complexity?

**Your answer:**

### E.1 What does the network use?

A neural network has no coefficients that can be read directly. A simple, model-agnostic way of measuring the importance of a feature is **permutation importance**: shuffle one column of the test set (breaking its relation with the target, while keeping its distribution) and measure how much the accuracy drops.

Compute the drop in test accuracy of the final model when each of these columns is shuffled: `Elevation`, `Horizontal_Distance_To_Roadways`, `Horizontal_Distance_To_Fire_Points`, `Hillshade_Noon` and `Aspect`. Use `np.random.RandomState(SEED).permutation` on a **copy** of `A_test` (the column index is `NUMERIC.index(name)`).

In [ ]:
importance = {}
for name in ["Elevation", "Horizontal_Distance_To_Roadways", "Horizontal_Distance_To_Fire_Points", "Hillshade_Noon", "Aspect"]:
    ...  # TODO

pd.Series(importance).sort_values(ascending=False)

In [ ]:
check(len(importance) == 5 and max(importance, key=importance.get) == "Elevation", "Elevation is the most important of the five")

**Explain:** Does the ranking agree with what you saw in the exploration? Why do the distances to roads and to fire points matter, although they have no direct effect on which trees grow? (Think about what else they could be measuring.) Why is a large importance not the same as a **causal** effect?

**Your answer:**

# Summary

Run the cell below and do not edit it. Your teacher can reproduce these numbers from your student number. It also shows the **question you must answer** in the cell after it.

In [ ]:
QUESTIONS = [
    "Using the backpropagation equations of Part A and your experiment of A.3, explain why the weights of a network must be initialised at random, and what would happen in your 3x512 network if all its weights started equal.",
    "Using your curves from D.1, explain why the validation loss went up while the validation accuracy did not go down, and which of the two you would use for early stopping.",
    "Using your results from C.3, explain which bug of the AI-generated code was the hardest to detect and why passing all the checks does not prove that the code is correct.",
    "Using your results from D.3, explain the trade-off between accuracy and balanced accuracy when you weight the loss, and which metric you used to choose your final model.",
]
print("Student:", STUDENT_ID)
print(pd.DataFrame(results).T.round(4).to_string())
print("Test:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in final.items()})
print("\nYour question:", QUESTIONS[STUDENT_ID % len(QUESTIONS)])

**Your answer:**

### AI usage declaration

If you used an AI tool (ChatGPT, Copilot, Gemini, Claude, ...) in this exercise, say **which one**, **for which sections**, and **what you asked it**. Give **one concrete example** of a piece of generated code (or explanation) that was wrong, incomplete or that you had to change, and how you noticed it. You may be asked to explain or change any cell of this notebook in class.

**Your answer:**